# Computational Set 4: Entangling Operations through Strong Light-Matter Coupling
[Jay Foley, Associated Professor of Chemistry, University of North Carolina Charlotte](https://foleylab.github.io/)

[Deputy Project Director of the Center for **Ma**ny-Body Methods, Spectroscopies, and Dynamics for Molecular **Pol**aritonic Systems](https://mapol-chem.github.io/#/)

Other open-source projects we contribute to:

- [**E**nhancing **S**cience **C**ourse by **I**ntegrating **P**thon (ESCIP)](https://escip.io/intro.html])
- [Molecular Sciences Software Institute Tutorials and Workshops](https://education.molssi.org/resources.html#programming)
- [Psi4Numpy Reference Implementations of Quantum Chemistry Methods](https://github.com/psi4/psi4numpy)

![Atom-Cavity System](images/cavity_atom_schematic.png)

Image of two atoms being entangled by photons confined by an optical cavity taken from [this highlight](https://physics.aps.org/articles/v10/s58) of the paper *Cavity Carving of Atomic Bell States* from [Phys. Rev. Lett. 118, 210503 (2017)](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.118.210503).


# 🧮 Part 1: Physical Model, Basis States, and Tensor Products

## Defining the Hamiltonian

## 🎯 Learning Outcomes
By the end of this part, you should be able to:
- Explain the terms in the Hamiltonian for two qubits (two independent two-level systems) coupled to a quantum light states of an optical cavity (Tavis–Cummings model).
- Explain how cavity-mediated coupling enables excitation transfer between two qubits
- Represent the Hamiltonian in a tensor product basis between uncoupled qubit and cavity states
- Understand how coupling two qubits to quantum light states in an optical cavity can facilitate the generation of entangled states known as Bell states
- Use the Design Recipe to compose a small toolkit of reusable functions

---


# 🧰 Building Your Toolkit with the Design Recipe

Recall from [Computational Set 1](spin_one_half.ipynb) that we build functions using the **Design Recipe**: Header → Purpose → Examples → Body → Test → Debug/Iterate.

In this notebook you'll use the recipe again to author a small toolkit of reusable functions — `kron3`, `density_matrix`, `commutator`, `compute_expectation_value`, `liouville_rhs`, and `rk4_step` — and then spend the second half of the notebook *using* that toolkit to simulate the iSWAP protocol and generate an entangled Bell state. A few helper functions that are pure bookkeeping (printing and formatting) are given to you pre-written, clearly labeled as such.


## 🧮 Representing the states of our system
We will represent the states of our system in a basis of uncoupled qubits and photon occupation states.

The basis states for each individual qubit are as follows:

$$
|g\rangle =
\begin{bmatrix}
1 \\
0
\end{bmatrix}
$$

$$
|e\rangle =
\begin{bmatrix}
0 \\
1
\end{bmatrix},
$$
and similarly for the cavity,

$$
|0\rangle =
\begin{bmatrix}
1 \\
0
\end{bmatrix}
$$

$$
|1\rangle =
\begin{bmatrix}
0 \\
1
\end{bmatrix},
$$

where we truncate cavity states at $|1\rangle$. Physically, you can think of these cavity states as defining the number of photons occupying the cavity, where $|0\rangle$ represent an empty cavity with 0 photons in it, $|1\rangle$ represents a cavity with 1 photon in it, and so on.
The **uncoupled basis** for the cavity and two qubits can be represented as *tensor products* of these individual basis states

$$
\begin{align}
|0\rangle \otimes |g\rangle \otimes |g\rangle, \\
|0\rangle \otimes |g\rangle \otimes |e\rangle, \\
|0\rangle \otimes |e\rangle \otimes |g\rangle, \\
|0\rangle \otimes |e\rangle \otimes |e\rangle, \\
|1\rangle \otimes |g\rangle \otimes |g\rangle, \\
|1\rangle \otimes |g\rangle \otimes |e\rangle, \\
|1\rangle \otimes |e\rangle \otimes |g\rangle, \\
|1\rangle \otimes |e\rangle \otimes |e\rangle.
\end{align}
$$
---
where $\otimes$ denotes a tensor product.

### Tensor Product Example

As an example, the tensor product between the cavity state $|0\rangle$ and the qubit ground state $|g\rangle$ is given by:

$$
|0\rangle \otimes |g\rangle =
\begin{bmatrix}
1 \\
0
\end{bmatrix}
\otimes
\begin{bmatrix}
1 \\
0
\end{bmatrix}
=
\begin{bmatrix}
1 \times
\begin{bmatrix}1 \\ 0\end{bmatrix} \\
0 \times
\begin{bmatrix}1 \\ 0\end{bmatrix}
\end{bmatrix}
=
\begin{bmatrix}
1 \\
0 \\
0 \\
0
\end{bmatrix}.
$$

This corresponds to the joint state where both the cavity and qubit are in their respective ground states.


Next we will introduce the Tavis-Cummings model that includes an interaction between each qubit and the cavity.



## 🔬 The Tavis–Cummings Hamiltonian
For two (or more) qubits  coupled to an $N$-level cavity:  

$$
H = \sum_i -\frac{\hbar \omega_i}{2} \sigma_{z,i} + \hbar \omega a^\dagger a 
   + \sum_i \hbar g_i ( a \sigma_{+,i} + a^\dagger \sigma_{-,i} )
$$

---

### ⚡ Term 1: Bare qubit Hamiltonian

Each two-level system (qubit) has an energy splitting described by:

$$
H_{\text{qubit},i} = -\frac{\hbar \omega_i}{2} \, \sigma_{z,i}
$$

with Pauli matrix:

$$
\sigma_z =
\begin{bmatrix}
1 & 0 \\
0 & -1
\end{bmatrix}
$$

So for one qubit:

$$
H_{\text{qubit}} =
\begin{bmatrix}
-\frac{\hbar\omega}{2} & 0 \\
0 & \frac{\hbar \omega}{2}
\end{bmatrix}
$$

---

### 📡 Term 2: Bare cavity Hamiltonian

The free Hamiltonian of the cavity mode is:

$$
H_{\text{cavity}} = \hbar \omega \, a^\dagger a
$$

If we truncate the cavity state space to $|0\rangle, |1\rangle$:

$$
a^\dagger a =
\begin{bmatrix}
0 & 0 \\
0 & 1
\end{bmatrix}
$$

So:

$$
H_{\text{cavity}} =
\begin{bmatrix}
0 & 0 \\
0 & \hbar \omega
\end{bmatrix}
$$

---

### 🔄 Term 3: Interaction Hamiltonian

The interaction term couples the cavity photons to the qubits through the exchange of an excitation.  

$$
H_{\text{int}} = \hbar g \, \left( a \, \sigma_{+} + a^\dagger \, \sigma_{-} \right)
$$

where we can interpret the operator $\sigma_{+}$ as an operator that takes the qubit from the ground to excited state, $\sigma_{-}$ takes the qubit from the excited to ground state, $a^\dagger$ excites the cavity by one photon, and $a$ de-excites the cavity by one photon.  The matrix form of these operators follows:

$$
\sigma_- =
\begin{bmatrix}
0 & 1 \\
0 & 0
\end{bmatrix}, \qquad
\sigma_+ =
\begin{bmatrix}
0 & 0 \\
1 & 0
\end{bmatrix}
$$

> **⚠️ Note on conventions:** **Here $\sigma_+$ raises the *energy* of the qubit by changing the ground state $|g\rangle = $ $\begin{pmatrix}1\\0\end{pmatrix}$ into the excited state $|e\rangle = $ $\begin{pmatrix}0\\1\end{pmatrix}$. This is the reverse of the convention in the [spin-1/2 lecture](../lectures/qm_postulates_two_level.ipynb), where $\hat{S}_+$ raises $m_s$ by changing $|\beta\rangle = $ $\begin{pmatrix}0\\1\end{pmatrix}$ into $|\alpha\rangle = $ $\begin{pmatrix}1\\0\end{pmatrix}$. Because $|g\rangle$ occupies the slot of $|\alpha\rangle$ and $|e\rangle$ occupies the slot of $|\beta\rangle$, the operators here are related to the spin ladder operators by $\sigma_+ = \frac{1}{\hbar}\hat{S}_+^{\dagger} = \frac{1}{\hbar}\hat{S}_-$ and $\sigma_- = \frac{1}{\hbar}\hat{S}_-^{\dagger} = \frac{1}{\hbar}\hat{S}_+$.**
>
> Two consequences to watch for:
> - $\sigma_z$ is the same matrix in both conventions, so $\sigma_z|g\rangle = +|g\rangle$: the *ground* state has $\sigma_z = +1$. That is why the qubit Hamiltonian carries a minus sign, $H_{\text{qubit}} = -\frac{\hbar\omega}{2}\sigma_z$.
> - Here $[\sigma_+, \sigma_-] = -\sigma_z$, whereas the lecture convention gives $[\hat{S}_+, \hat{S}_-] = 2\hbar\hat{S}_z$ (i.e., $+\sigma_z$). You will see the $-\sigma_z$ result when you test your `commutator` function below.

and the cavity ladder operators truncated to $ |0\rangle, |1\rangle $:

$$
a =
\begin{bmatrix}
0 & 1 \\
0 & 0
\end{bmatrix}, \qquad
a^\dagger =
\begin{bmatrix}
0 & 0 \\
1 & 0
\end{bmatrix}
$$

So the coupling block looks like:

$$
\hbar g (a \sigma_+ + a^\dagger \sigma_-) \quad \longrightarrow \quad
\begin{bmatrix}
0 & 0 & 0 & 0 \\
0 & 0 & \hbar g & 0 \\
0 & \hbar g & 0 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}
$$

(in the truncated $|0\rangle \otimes |g\rangle, |0\rangle \otimes |e\rangle, |1\rangle \otimes g\rangle, |1\rangle \otimes e\rangle$ basis).
Physically, you can think of this interaction as moving excitations between the qubit and the cavity!

---

✨ Putting all terms together gives the Tavis–Cummings Hamiltonian in block-matrix form, built from:
- **Qubit terms**: diagonal blocks ($-\hbar \omega/2 \, \sigma_z$)  
- **Cavity term**: photon number operator ($\hbar \omega a^\dagger a$)  
- **Interaction**: off-diagonal couplings that exchange excitations between the cavity and qubit(s).  

---

## ⚡ Physical Picture: Generating Bell states via cavity coupling
We model the following process to evolve an initial factorizable state to a non-factorizable (entangled) Bell state

1. **Initial excitation**: At $t=0$, molecule 1 is excited and coupled to the cavity,
   
   $$|\Psi(t=0)\rangle = |0\rangle \otimes |e\rangle \otimes |g\rangle$$
   
2. **Excitation transfer**: Over time $T_1$, molecule 1 exchanges energy with the cavity,
 
 $$|\Psi(t=T_1)\rangle = \left( c_{0e} |0\rangle \otimes |e\rangle + c_{1g} |1\rangle \otimes |g\rangle \right) \otimes |g\rangle $$

3. **Switch coupling**: At $T_1$, molecule 1 decouples and molecule 2 couples.  
4. **Final exchange**: By $T_2$, molecule 2 has exchanged energy with the cavity.  

 $$|\Psi(t=T_2)\rangle = |0\rangle \otimes \left( c_{eg} |e\rangle \otimes |g\rangle + c_{ge} |g\rangle \otimes |e\rangle \right)$$

If $T_1$ and $T_2$ are designed properly, the qubits end in a **Bell state**:

$$
|\Psi(t_f)\rangle = \sqrt{\tfrac{1}{2}} \left( |g\rangle \otimes |e\rangle \;-\; |e\rangle \otimes |g\rangle \right)
$$

---


## 🧮 Building the Basis States

Now we’ll explicitly construct the **uncoupled basis states** by taking tensor products of the cavity states $|0\rangle, |1\rangle$ and the qubit states $|g\rangle, |e\rangle$.

🎯 Learning outcomes:
- Represent cavity and molecular states as column vectors.  
- Use tensor products (via `np.kron(A,B)`, which performs the Kronecker product between `A` and `B`, equivalent to ${\bf A} \otimes {\bf B}$ ) to build composite states.  
- Confirm the basis matches the order written in the previous section.  


In [ ]:
import numpy as np
import numpy.linalg as la

# Define the Hamiltonian parameters
omega_0 = 1.0  # Atomic frequency (in arbitrary units)
omega = 1.0    # Cavity frequency (in arbitrary units)
g = 0.1        # Coupling strength (in arbitrary units)
hbar = 1.0     # Reduced Planck's constant (set to 1 for simplicity)

# Gate times
T1_time = np.pi / (4 * g)
T2_time = 2 * np.pi / (4 * g)


# --- Basis vectors for individual subsystems ---
z_ket = np.array([[1], [0]])  # |0> cavity state
o_ket = np.array([[0], [1]])  # |1> cavity state

g_ket = np.array([[1], [0]])  # |g> ground state of a molecule
e_ket = np.array([[0], [1]])  # |e> excited state of a molecule


## 🧩 Design Recipe: `kron3` — Building Three-Way Tensor Products

Every composite state and operator in this notebook lives in the 8-dimensional space `cavity ⊗ qubit 1 ⊗ qubit 2`. That means you'll be nesting `np.kron` calls like `np.kron(A, np.kron(B, C))` over and over. Let's write one small function, `kron3`, that does this nesting for us — then reuse it everywhere.

1. **Header** Given below: `kron3(A, B, C)`, taking three arrays (kets or operators) and returning their tensor product `A ⊗ B ⊗ C`.
2. **Purpose** Given below in the docstring.
3. **Examples** One example is given — add a second example of your own, using `o_ket`, `g_ket`, and `e_ket` in some combination.
4. **Body** Complete the line marked `...`. (Hint: `np.kron(A, B)` gives you `A ⊗ B`; you just need to nest another `np.kron` inside.)
5. **Test** Run the test cell below.
6. **Debug/Iterate** If a test fails, check that you passed `A`, `B`, `C` to `np.kron` in the right positions — the first argument should end up in the "outermost" (leftmost) block of the result.


In [ ]:
def kron3(A, B, C):
    """
    Compute the tensor (Kronecker) product of three subsystem states or operators.

    Arguments
    ---------
    # TODO: Add arguments line

    Returns
    -------
    # TODO: Add return line

    Examples
    --------
    np.allclose(kron3(z_ket, g_ket, g_ket), np.array([[1],[0],[0],[0],[0],[0],[0],[0]])
    # TODO: Add one more example of your own.
    """
    # TODO: Return the tensor product of A, B, and C
    result = ...
    return result


In [ ]:
assert np.allclose(kron3(z_ket, g_ket, g_ket).ravel(), [1, 0, 0, 0, 0, 0, 0, 0])
assert np.argmax(kron3(o_ket, g_ket, g_ket)) == 4
assert kron3(np.eye(2), np.eye(2), np.eye(2)).shape == (8, 8)

# TODO: Add an assert statement for the second Example from your kron3 docstring, the one you added!

print("✅ kron3 tests passed!")


In [ ]:
# --- Tensor products to build composite states ---
# ✏️ TODO 1: Define composite states using kron3(left_state, middle_state, right_state)
# Example pattern: zgg_ket = kron3(z_ket, g_ket, g_ket)
zgg_ket = ...  # |0> ⊗ |g> ⊗ |g>
zge_ket = ...  # |0> ⊗ |g> ⊗ |e>
zeg_ket = ...  # |0> ⊗ |e> ⊗ |g>
zee_ket = ...  # |0> ⊗ |e> ⊗ |e>

ogg_ket = ...  # |1> ⊗ |g> ⊗ |g>
oge_ket = ...  # |1> ⊗ |g> ⊗ |e>
oeg_ket = ...  # |1> ⊗ |e> ⊗ |g>
oee_ket = ... # |1> ⊗ |e> ⊗ |e>

# --- Optional: Insert Print Statements to inspect one or more of the composite ket states ---


### ✍️ Required responses for Part 1

Answer each question in complete sentences.

1. Why does the complete cavity–qubit–qubit basis contain eight states? Explain using the dimensions of the three subsystems.

   **Response 1:** Replace this text with your answer.

2. How does the ordering `cavity ⊗ qubit 1 ⊗ qubit 2` determine the position of $|1gg\rangle$ in the eight-element state vector?

   **Response 2:** Replace this text with your answer.

3. Which basis states belong to the single-excitation subspace? Why does Tavis–Cummings evolution beginning from $|0eg\rangle$ remain within that subspace?

   **Response 3:** Replace this text with your answer.


# 🧮 Part 2: Operator Algebra and Composite-Space Operators

## Local operator matrices

We will now set up the qubit and cavity operators in Python. The definitions of the key matrices are

$$
\sigma_z =
\begin{bmatrix}
1 & 0 \\
0 & -1
\end{bmatrix},
$$

$$
\sigma_- =
\begin{bmatrix}
0 & 1 \\
0 & 0
\end{bmatrix}, \qquad
\sigma_+ =
\begin{bmatrix}
0 & 0 \\
1 & 0
\end{bmatrix},
$$

and

$$
a =
\begin{bmatrix}
0 & 1 \\
0 & 0
\end{bmatrix}, \qquad
a^\dagger =
\begin{bmatrix}
0 & 0 \\
1 & 0
\end{bmatrix}.
$$

We can build NumPy arrays corresponding to these matrices using the syntax demonstrated for $\sigma_z$:

```python
sigma_z = np.array([[1, 0], [0, -1]])
```


In [ ]:
# 🧩 Step: Define Pauli and ladder operators in matrix form

# --- Pauli matrices for the two-level system (qubit / molecule) ---

# ✏️ TODO 1: Define sigma_z using np.array([[1, 0], [0, -1]])
sigma_z = ...

# ✏️ TODO 2: Define the lowering operator sigma_minus
# (Hint: sigma_- = [[0, 1], [0, 0]])
sigma_minus = ...

# ✏️ TODO 3: Define the raising operator sigma_plus
# (Hint: sigma_+ = [[0, 0], [1, 0]])
sigma_plus = ...
# --- Ladder operators for the truncated cavity (|0>, |1> basis) ---

# ✏️ TODO 4: Define the annihilation operator 'a'
# (Hint: a = [[0, 1], [0, 0]])
a = ...

# ✏️ TODO 5: Define the creation operator 'adag'
# (Hint: a† = [[0, 0], [1, 0]])
adag = ...

# ✅ Quick sanity check
print("sigma_z:\n", sigma_z)
print("sigma_minus:\n", sigma_minus)
print("sigma_plus:\n", sigma_plus)
print("a:\n", a)
print("adag:\n", adag)


## 🧩 Design Recipe: `commutator` — Testing Compatibility of Operators

The commutator $[A, B] = AB - BA$ shows up constantly when checking algebraic relationships between operators — you're about to use it to verify the Pauli and ladder-operator relations below. Let's write one function for it instead of retyping `A @ B - B @ A` every time.

1. **Header** Given below: `commutator(A, B)`, taking two square matrices and returning their commutator.
2. **Purpose** Given below in the docstring.
3. **Examples** One example is given — add a second example of your own, using operators you've already defined (e.g. `sigma_plus`, `sigma_minus`, `a`, `adag`).
4. **Body** Complete the line marked `...`.
5. **Test** Run the test cell below.
6. **Debug/Iterate** If a test fails, double check the order of subtraction — `[A, B]` is `AB - BA`, not `BA - AB`.


In [ ]:
def commutator(A, B):
    """
    Compute the commutator of two operators, [A, B] = AB - BA.

    Arguments
    ---------
    # TODO: Add arguments line

    Returns
    -------
    # TODO: Add return line

    Examples
    --------
    commutator(a, adag) == [[1, 0], [0, -1]]
    # TODO: Add one more example of your own
    """
    # TODO: Return the commutator of A and B
    result = ...
    return result


In [ ]:
assert np.allclose(commutator(a, adag), [[1, 0], [0, -1]])

# TODO: Complete a second test using the qubit ladder operators.
# With the convention used here, [sigma_plus, sigma_minus] = -sigma_z.

assert np.allclose(commutator(sigma_plus, sigma_minus), ...)
print("✅ commutator tests passed!")


In [ ]:
# 🧪  Verify the properties of your operators by running this block

# --- Pauli operator relationships ---
print("🧭 Checking Pauli operator relationships...\n")

# Identity from Pauli matrices
identity_qubit = np.eye(2)

# ✏️ TODO 1: Verify that σ₊σ₋ + σ₋σ₊ = I (the identity matrix)
lhs_pauli = ...
print("σ₊σ₋ + σ₋σ₊ =\n", lhs_pauli)
assert np.allclose(lhs_pauli, identity_qubit), "❌ Check your sigma_plus/sigma_minus definitions!"

# ✏️ TODO 2: Use your commutator function to compute [σ₊, σ₋]
comm_pauli = ...
print("\n[σ₊, σ₋] =\n", comm_pauli)
print("Expected result: -σ_z\n")

# --- Cavity operator relationships ---
print("💡 Checking cavity ladder operator relationships...\n")

# ✏️ TODO 3: Verify that a†a is the number operator
num_op = ...
print("a†a =\n", num_op)
print("Expected result:\n[[0, 0], [0, 1]]\n")

# ✏️ TODO 4: Use your commutator function to compute [a, a†]
comm_cavity = ...
print("[a, a†] =\n", comm_cavity)
print("Expected result:\n[[1, 0], [0, -1]] (i.e., the Pauli σ_z equivalent for 2-level truncation)\n")

# ✅ If no assertion failed, all is good!
print("✅ All operator checks passed! 🎉")


## 🧱 Building the Hamiltonian in the Composite Hilbert Space

Now we’ll extend the operators you defined earlier into the **full tripartite Hilbert space**, which is the tensor product of:


$\mathscr{H}_{\text{total}} = \mathscr{H}_{\text{cavity}} \otimes \mathscr{H}_{\text{qubit 1}} \otimes \mathscr{H}_{\text{qubit 2}} $


Each subsystem is **two-dimensional** (truncated to $|0\rangle, |1\rangle$ for the cavity and $|g\rangle, |e\rangle$ for each molecule), so the total dimension is $2 \times 2 \times 2 = 8$.

The **total Hamiltonian**, however, is a **sum of tensor-product–extended operators**, not a tensor product of Hamiltonians:


$H_{\text{total}} = H_{\text{cavity}} \otimes I_1 \otimes I_2 + I_{\text{cav}} \otimes H_1 \otimes I_2 + I_{\text{cav}} \otimes I_1 \otimes H_2 + H_{\text{int}} $

Each term acts only on its subsystem but lives in the full composite space.

---

### ⚙️ Step 1: Identities for Each Subspace
We start by defining the $2\times2$ identity matrices for the cavity and for a generic qubit:

$$
I_{\text{cav}} = I_{\text{q}} = 
\begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

These act as “placeholders” when we want an operator to act on one subsystem but leave the others unchanged.

---

### 🧩 Step 2: Tensor-Product Construction of Operators
We build operators for each molecule and for the cavity in the **composite space** using the Kronecker product (`np.kron`):

For **qubit 1**:
$$
  \sigma_{z,1} = I_{\text{cav}} \otimes \sigma_z \otimes I_{\text{q}}, \quad
  \sigma_{+,1} = I_{\text{cav}} \otimes \sigma_+ \otimes I_{\text{q}}, \quad
  \sigma_{-,1} = I_{\text{cav}} \otimes \sigma_- \otimes I_{\text{q}}
$$

For **qubit 2**:
$$
  \sigma_{z,2} = I_{\text{cav}} \otimes I_{\text{q}} \otimes \sigma_z, \quad
  \sigma_{+,2} = I_{\text{cav}} \otimes I_{\text{q}} \otimes \sigma_+, \quad
  \sigma_{-,2} = I_{\text{cav}} \otimes I_{\text{q}} \otimes \sigma_-
$$

For the **cavity**:
$$
  a = a \otimes I_{\text{q}} \otimes I_{\text{q}}, \qquad
  a^\dagger = a^\dagger \otimes I_{\text{q}} \otimes I_{\text{q}}
$$


This ensures that each operator acts only on its subsystem while preserving the full system dimension.

---


In [ ]:
# create identities for each Hilbert space
Icav = np.eye(2)
Iq   = np.eye(2)

# 🧩  Build operators in the composite (tripartite) Hilbert space
# The total space is:  cavity ⊗ qubit 1 ⊗ qubit 2

# Hints:
# - Use the kron3(A, B, C) function you built above to take the tensor product
# - When an operator should act only on one subsystem,
#   multiply it by identity matrices on the other two
# - Notice you are reusing the function you just built 8 more times!

# --- Operators for molecule (qubit) 1 ---
# ✏️ TODO 1: Define sigma_z for qubit 1
# sz1 = kron3( ..., ..., ... )
sz1 = ...

# ✏️ TODO 2: Define sigma_plus for qubit 1
# sp1 = kron3( ..., ..., ... )
sp1 = ...

# ✏️ TODO 3: Define sigma_minus for qubit 1
# sm1 = kron3( ..., ..., ... )
sm1 = ...

# --- Operators for molecule (qubit) 2 ---
# ✏️ TODO 4: Define sigma_z for qubit 2
# sz2 = kron3( ..., ..., ... )
sz2 = ...

# ✏️ TODO 5: Define sigma_plus for qubit 2
# sp2 = kron3( ..., ..., ... )
sp2 = ...

# ✏️ TODO 6: Define sigma_minus for qubit 2
# sm2 = kron3( ..., ..., ... )
sm2 = ...

# --- Operators for the cavity mode ---
# ✏️ TODO 7: Define the annihilation operator (a) in full space
# am = kron3( ..., ..., ... )
am = ...

# ✏️ TODO 8: Define the creation operator (a† = adag) in full space
# ap = kron3( ..., ..., ... )
ap = ...

# 💡 Hint:
# For the cavity operators, they should act only on the cavity subspace,
# so they go in the first argument of kron3, followed by identities for both qubits.
#
# For qubit 1, the operator acts on the middle argument of kron3.
# For qubit 2, it acts on the last argument of kron3.


### ✍️ Required responses for Part 2

Answer each question in complete sentences.

1. Why are identity matrices required when extending a local operator into the composite Hilbert space? Compare `sp1` and `sp2` in your answer.

   **Response 1:** Replace this text with your answer.

2. Describe physically what $a\sigma_{+,1}$ and $a^\dagger\sigma_{-,1}$ do to the cavity–qubit system.

   **Response 2:** Replace this text with your answer.

3. Why does the two-level cavity truncation produce $[a,a^\dagger]=\mathrm{diag}(1,-1)$ rather than the infinite-dimensional result $I$?

   **Response 3:** Replace this text with your answer.


# 🧮 Part 3: Composite Hamiltonians and Quantum States

## Hamiltonian Terms
Using these building blocks, we define:

- **Bare energies**:
$$
  -\frac{\hbar \omega_0}{2}(\sigma_{z,1} + \sigma_{z,2}) 
  + \hbar \omega\, a^\dagger a
$$
  which give the unperturbed qubit and cavity energies.

- **Interaction terms**:
$$
  \hbar g \left(a\,\sigma_{+,i} + a^\dagger\,\sigma_{-,i}\right)
$$
  which represents the exchange of an excitation between the cavity and qubit $i$.

---

### 🔁 Step 4: Two Coupling Configurations
We’ll define:
- **$H_1$** — only molecule 1 coupled to the cavity (molecule 2 off).  
- **$H_2$** — only molecule 2 coupled to the cavity (molecule 1 off).  
- **$H$** — both molecules coupled simultaneously (used later).

These Hamiltonians let us simulate the **iSWAP protocol**, where coupling is “switched” between molecules in time.

---

💡 *Tip:*  
Try printing the shapes of `H1`, `H2`, and `H` — they should each be $8\times8$ Hermitian matrices.

In [ ]:
# 🧮 Step: Build the composite Hamiltonians

# The total Hamiltonian acts on the full 8×8 composite space:
#   cavity ⊗ qubit 1 ⊗ qubit 2
# We’ll build:
#   - H1 : only molecule 1 coupled to the cavity
#   - H2 : only molecule 2 coupled to the cavity
#   - H  : both molecules coupled (not used here, but useful later)

# --- Example: H1 (Molecule 1 coupled) ---
# This Hamiltonian includes:
#   1. Bare qubit energies (σ_z terms)
#   2. Bare cavity energy (a†a term)
#   3. Coupling between the cavity and molecule 1 (a σ_+1 + a† σ_-1)
#   Note: We use * for scalar multiplication and for multiplying a matrix by a scalar
#         We use @ for matrix-matrix multiplication

H1 = -hbar * omega_0 / 2 * sz1        # qubit 1 Hamiltonian
H1 -=  hbar * omega_0 / 2 * sz2        # qubit 2 Hamiltonian
H1 +=  hbar * omega * ap @ am          # cavity Hamiltonian
H1 +=  hbar * g * ( am @ sp1 + ap @ sm1 )  # coupling: cavity ↔ molecule 1

print("✅ H1 constructed successfully! Dimension:", H1.shape)

# --- ✏️ TODO: Build H2 (Molecule 2 coupled) ---
# 💡 HINTS:
# - Start from a copy of H1 (use np.copy())
# - Add coupling terms for molecule 2: (a σ_+₂ + a† σ_-₂)
# - Remove coupling terms for molecule 1 (subtract them)

H2 = ... # start from H1
H2 += ... # add coupling for molecule 2
H2 -= ... # subtract coupling for molecule 1


# --- ✏️ TODO: Build H (Both molecules coupled) ---
# 💡 HINTS:
# - Start again from H1, which already contains the molecule 1 coupling
# - Add the molecule 2 coupling term
# - This represents the case when both qubits are simultaneously interacting with the cavity

H = np.copy(H1)
H += hbar * g * ( am @ sp2 + ap @ sm2 )

# --- Optional sanity checks ---
print("H2 shape:", H2.shape)
print("H shape:", H.shape)
print("Hermitian check H1:", np.allclose(H1, H1.conj().T))
print("Hermitian check H2:", np.allclose(H2, H2.conj().T))


## 🌱 Step 5: Defining the Initial State

We’ll start with the system in a **single-excitation configuration**:  
- The **cavity** is empty $(|0\rangle$),  
- **Qubit 1** is excited ($|e\rangle$),  
- **Qubit 2** is in its ground state ($|g\rangle$).  

$$
|\psi(0)\rangle = |0\rangle_{\text{cav}} \otimes |e\rangle_1 \otimes |g\rangle_2
$$

---

### 🧮 In the Uncoupled Basis
The corresponding **density matrix** is formed as the outer product:
$$
\rho(0) = 
\left(|0\rangle \otimes |e\rangle \otimes |g\rangle\right)
\left(\langle 0| \otimes \langle e| \otimes \langle g|\right)
$$
This state is often written compactly as:
$$
\rho(0) = |0eg\rangle \langle 0eg|
$$
and represents a *pure state localized on molecule 1*.

---

### 💡 Physical Picture
- At $t=0$, **the energy resides entirely in qubit 1**.  
- The cavity and qubit 2 begin in their **ground states**.  
- As time evolves, this excitation can be exchanged through the cavity mode — 
  forming superpositions that resemble hybrid light-matter states called **polaritons**.

---


> 💭 **Checkpoint Question:**  
> What physical process causes population transfer between $|0eg\rangle$ and $|0ge\rangle$ during the iSWAP sequence?
>
> <details>
> <summary><strong>Hint</strong> (click to expand)</summary>
> The excited qubit can interact with the cavity vacuum ($|0\rangle$) to stimulate the emission of a photon into the cavity.  Similarly, an occupied cavity ($|1\rangle$) can interact with a qubit in the ground state to stimulate absorption of a photon.
> </details>


## 🧩 Design Recipe: `density_matrix` — From a Ket to a Density Matrix

We're about to build our initial state as a density matrix, $\rho(0) = |\psi(0)\rangle\langle\psi(0)|$, and later we'll need the same construction again for our target Bell state. Let's write a function for it once.

1. **Header** Given below: `density_matrix(ket)`, taking a normalized column vector and returning the pure-state density matrix $|\psi\rangle\langle\psi|$.
2. **Purpose** Given below in the docstring.
3. **Examples** None are given this time — write at least 2 examples of your own in the docstring, using kets you've already defined (e.g. `g_ket`, `e_ket`).
4. **Body** Complete the line marked `...`. (Hint: $|\psi\rangle\langle\psi|$ is an outer product — the bra is the conjugate transpose of the ket.)
5. **Test** We'll test your function once we've built the initial state `psi0` a little further down, so that we have a concrete state to test against.
6. **Debug/Iterate** If your trace test fails, double check the order of the outer product — `ket @ ket.conj().T`, not `ket.conj().T @ ket` (that would give you back the bra-ket instead!).


In [ ]:
def density_matrix(ket):
    """
    Compute the pure-state density matrix |psi><psi| for a normalized ket.

    Arguments
    ---------
    # TODO: Add arguments line

    Returns
    -------
    # TODO: Add return line

    Examples
    --------
    # TODO: Add at least 2 examples of your own, e.g.:
    # density_matrix(g_ket) == ...
    # add example 2 here
    """
    # TODO: Return the outer product of ket with its own conjugate transpose
    rho = ...
    return rho


In [ ]:
# 🧩 Create an initial state corresponding to |0> ⊗ |e> ⊗ |g>

# Recall:
# |0>_cavity  = z_ket
# |1>_cavity  = o_ket
# |g>_qubit   = g_ket
# |e>_qubit   = e_ket

# ✏️ TODO 1: Assign psi0 to the appropriate basis vector from our first code block
psi0 = ...


# ✅ Step: Construct the density matrix ρ = |ψ⟩⟨ψ| using your density_matrix function
rho0 = density_matrix(psi0)

# --- Sanity checks ---
print("Shape of psi0:", psi0.shape)
print("Norm squared:", np.vdot(psi0, psi0))
print("Density matrix shape:", rho0.shape)
print("Trace(ρ) =", np.trace(rho0))   # should be 1 for a pure normalized state

# 💡 Tip:
# Try visualizing which basis states have nonzero amplitudes.
# Since this is |0eg>, only one entry in the 8-element state vector should be 1.


In [ ]:
# ✅ Now that psi0 is defined, test density_matrix against it
assert np.allclose(density_matrix(g_ket), [[1, 0], [0, 0]])
assert np.allclose(density_matrix(e_ket), [[0, 0], [0, 1]])
assert np.isclose(np.trace(density_matrix(psi0)).real, 1.0)
print("✅ density_matrix tests passed!")


## 🧪 Design Recipe: `compute_expectation_value` — Recall, then Discover

### Recall
Back in [Computational Set 1](spin_one_half.ipynb), you computed an expectation value $\langle\psi|\hat O|\psi\rangle$ as a bra-ket sandwich: `bra @ Operator @ ket`. Here our states are column vectors (`state`), so the direct analogue is:

```python
state.conj().T @ H @ state
```

### Notice a wrinkle
Try that expression on a couple of states and operators. You'll notice it hands you back a `(1, 1)` numpy array, not a clean scalar — because `state` is a column vector, not a flat 1D array. Is there a single NumPy operation that computes this same conjugated sandwich, $\langle\text{state}|H|\text{state}\rangle$, and hands back a plain scalar? Look through the NumPy documentation for something that conjugates its first argument automatically — you may find `np.vdot` interesting.

Either use `np.vdot(state, H @ state)`, or apply `.item()` to the `(1, 1)` array produced by the explicit bra-ket sandwich. In either case, your function must return a scalar.

1. **Header** Given below: `compute_expectation_value(state, H)`.
2. **Purpose** Given below in the docstring.
3. **Examples** One example is given — add a second example of your own.
4. **Body** Complete the line marked `...`.
5. **Test** Run the test cell below.
6. **Debug/Iterate** If your result prints as `[[...]]` instead of a bare number, convert the result to a scalar with `.item()` or use `np.vdot`.


In [ ]:
def compute_expectation_value(state, H):
    """
    Compute the expectation value <state|H|state> for a state vector and operator.

    Arguments
    ---------
    # TODO: Add arguments line

    Returns
    -------
    # TODO: Add return line

    Examples
    --------
    compute_expectation_value(o_ket, adag @ a) == 1
    # TODO: Add one more example of your own, e.g. using z_ket or e_ket
    """
    # TODO: Return the expectation value of H in the given state as a scalar
    # (Recall the bra-ket sandwich pattern -- then see if you can find a
    #  single NumPy function that does the conjugation for you.)
    expectation = ...
    return expectation


In [ ]:
assert np.isclose(compute_expectation_value(o_ket, adag @ a), 1.0)
assert np.isclose(compute_expectation_value(z_ket, adag @ a), 0.0)
assert np.isclose(compute_expectation_value(e_ket, sigma_z), -1.0)
print("✅ compute_expectation_value tests passed!")

# 🔬 Now that it's built, let's actually use it: a one-line energy sanity check.
print("Energy expectation of the initial state:", compute_expectation_value(psi0, H1))
# For |0eg> under these parameters, this should come out to 0.0


### ✍️ Required responses for Part 3

Answer each question in complete sentences.

1. What is the difference between $H_1$, $H_2$, and $H$? Identify which interaction terms are present in each Hamiltonian.

   **Response 1:** Replace this text with your answer.

2. How do the trace, Hermiticity, and idempotency condition $\rho_0^2=\rho_0$ show that $\rho_0$ represents a normalized pure state?

   **Response 2:** Replace this text with your answer.

3. What does the initial energy expectation value mean physically? Which terms in the Hamiltonian contribute to it?

   **Response 3:** Replace this text with your answer.


# 🧮 Part 4: Density-Matrix Time Propagation

In the next section, we’ll simulate the time evolution of our system in the **uncoupled basis** by solving the **Liouville–von Neumann equation**:

$$
\frac{d\rho}{dt} = -\frac{i}{\hbar} [H, \rho]
$$

To get there we'll build two more functions with the Design Recipe — `liouville_rhs`, which computes the right-hand side of that equation, and `rk4_step`, which uses it to advance $\rho$ forward by one small time step. A few purely bookkeeping helper functions (printing and formatting) are given to you pre-written below, since they don't teach any new physics or numerics.


### 🧰 Pre-written helper functions (you do not need to write these)

`complex_formatter`, `print_operator_action`, and `represent_basis_in_eigen` are bookkeeping/display utilities — they don't compute any new physics, so they're given to you fully written. Feel free to read through them later, but there's no need to build them yourself.


In [ ]:
# simple function for printing real part of complex matrix
def complex_formatter(x):
    return f"{x.real: .2f}" #{x.imag:+.2f}j"

def print_operator_action(op, ket, op_name="Operator", ket_name="Ket"):
    """ prints the action of an operator on a ket in a formatted way """
    result = op @ ket

    # Formatters
    fmt = {'all': lambda x: f"{x: 6.3f}"}

    print(f"\n=== {op_name} acting on {ket_name} ===\n")

    print(f"{op_name} matrix in tensor product space:")
    print(np.array2string(op, formatter=fmt))

    print(f"\n{ket_name} vector in tensor product space:")
    print(np.array2string(ket.reshape(-1, 1), formatter=fmt))

    print(f"\nResulting vector in tensor product space ({op_name} \u22c5 {ket_name}):")
    print(np.array2string(result.reshape(-1, 1), formatter=fmt))

def represent_basis_in_eigen(basis_names, eigvecs, energies=None, tol=1e-6):
    """
    For each original basis state |b_j>, prints its expansion
      |b_j> = sum_i c_{j,i} |ψ_i>
    where c_{j,i} = <ψ_i|b_j> = conj(eigvecs[j, i]) for unitary eigvecs.

    Parameters
    ----------
    basis_names : list of str
        Names of the basis kets in the same order as the rows of eigvecs.
    eigvecs : np.ndarray, shape (N, N)
        Columns are the eigenvectors |ψ_i> expressed in the original basis.
    energies : array-like, optional
        If provided, labels the eigenstates by energy order.
    tol : float
        Threshold below which coefficients are treated as zero.
    """
    N = eigvecs.shape[0]
    assert eigvecs.shape == (N, N)
    assert len(basis_names) == N

    # If energies given, label states by E; otherwise by index
    labels = []
    if energies is not None:
        for i, E in enumerate(energies, 1):
            labels.append(f"ψ_{i} (E={E:.4f})")
    else:
        labels = [f"ψ_{i}" for i in range(1, N+1)]

    for j, name in enumerate(basis_names):
        # coefficients c_{j,i} = <ψ_i|b_j> = conj(eigvecs[j,i])
        coefs = np.conj(eigvecs[j, :])
        terms = []
        for i, c in enumerate(coefs):
            if abs(c) < tol:
                continue
            # format complex; drop imaginary part if ~0
            if abs(c.imag) < tol:
                terms.append(f"{c.real:+.4f}\u00b7|{labels[i]}>")
            else:
                terms.append(f"({c.real:+.4f}{c.imag:+.4f}j)\u00b7|{labels[i]}>")
        combo = "  +  ".join(terms) if terms else "0"
        print(f"|{name}> = {combo}\n")


## 🧩 Design Recipe: `liouville_rhs` — The Liouville–von Neumann Right-Hand Side

We need a function that computes $-\frac{i}{\hbar}[H, \rho]$, the right-hand side of the equation above. You already have a `commutator` function — reuse it here instead of re-deriving $H\rho - \rho H$ by hand.

1. **Header** Given below: `liouville_rhs(H, rho, hbar=1.0)`.
2. **Purpose** Given below in the docstring.
3. **Examples** None are given this time — write at least 1 example of your own in the docstring.
4. **Body** Complete the line marked `...`, calling your `commutator` function.
5. **Test** Run the test cell below.
6. **Debug/Iterate** If your Hermiticity check fails, double check the factor of $-i$ out front, and make sure you're passing `H` and `rho` to `commutator` in that order. (The right-hand side comes out **Hermitian**, not anti-Hermitian -- that's exactly what keeps $\rho$ a valid density matrix as it evolves.)


In [ ]:
def liouville_rhs(H, rho, hbar=1.0):
    """
    Compute the right-hand side of the Liouville-von Neumann equation,
    d(rho)/dt = -(i/hbar) [H, rho].

    Arguments
    ---------
    # TODO: Add arguments line

    Returns
    -------
    # TODO: Add return line

    Examples
    --------
    # TODO: Add at least 1 example of your own, e.g.:
    # liouville_rhs(np.eye(8), rho0) == np.zeros((8, 8))  (identity commutes with everything)
    """
    # TODO: Reuse your commutator function to compute -(i/hbar) * [H, rho]
    rhs = ...
    return rhs


In [ ]:
# If H commutes with rho, the RHS is zero (identity commutes with everything):
assert np.allclose(liouville_rhs(np.eye(8), rho0), np.zeros((8, 8)))

# The RHS is Hermitian whenever H and rho are Hermitian -- this is exactly why
# rho + dt * rhs stays a valid (Hermitian) density matrix as it evolves:
rhs = liouville_rhs(H1, rho0)
assert np.allclose(rhs, rhs.conj().T)
print("✅ liouville_rhs tests passed!")


## 🚧 Design Recipe: `rk4_step` — Advancing ρ One Time Step (Guided Template)

This is the last function you'll build, and it needs the most scaffolding, so let's walk through the idea first.

**The RK4 idea:** to advance $\rho$ by one step of size $dt$, we estimate the *slope* $d\rho/dt$ four times — once at the start of the interval ($k_1$), twice at the midpoint using increasingly refined estimates ($k_2$, $k_3$), and once at the end ($k_4$) — then take a weighted average of the four slopes to make the actual step. This is the same underlying concept you already exercised with **Newton–Raphson optimization** in the earlier geometry-optimization work: you *iterate a computed update rule* to march a quantity forward. There, the quantity was a molecular geometry and the update came from a gradient; here, the quantity is the density matrix $\rho$, the independent variable is time, and the update rule is `liouville_rhs` (which you just built).

For $f(\rho)=-(i/\hbar)[H,\rho]$, the fourth-order Runge–Kutta update is

$$
\begin{aligned}
k_1 &= f(\rho_n), \\
k_2 &= f\!\left(\rho_n+\frac{dt}{2}k_1\right), \\
k_3 &= f\!\left(\rho_n+\frac{dt}{2}k_2\right), \\
k_4 &= f\!\left(\rho_n+dt\,k_3\right), \\
\rho_{n+1} &= \rho_n+\frac{dt}{6}\left(k_1+2k_2+2k_3+k_4\right).
\end{aligned}
$$

The cell below gives you `k1` fully worked. Fill in `k2`, `k3`, `k4`, and the final weighted combination, reusing `liouville_rhs` at each step.

1. **Header** Given below: `rk4_step(rho, H, dt, hbar=1.0)`.
2. **Purpose** Given below in the docstring.
3. **Examples** One example is sketched (commented out) in the docstring — feel free to add your own.
4. **Body** `k1` is worked for you; fill in `k2`, `k3`, `k4`, and `rho_next`.
5. **Test** Run the test cell below.
6. **Debug/Iterate** If the trace or Hermiticity test fails, double check that each `k_i` steps `rho` forward using the *previous* slope (e.g. `k2` uses `rho + 0.5 * dt * k1`, not `rho + dt * k1`), and that the final combination weights are `1, 2, 2, 1` out of 6.


In [ ]:
def rk4_step(rho, H, dt, hbar=1.0):
    """
    Advance the density matrix by one RK4 time step under a
    time-independent Hamiltonian H.

    Arguments
    ---------
    rho  : the current density matrix
    H    : the (time-independent) Hamiltonian
    dt   : the time step
    hbar : reduced Planck's constant (default 1.0)

    Returns
    -------
    rho_next : the density matrix after one step of size dt

    Examples
    --------
    # With H = I, rho commutes with H, so rho is unchanged:
    # rk4_step(rho0, np.eye(8), 0.05) == rho0
    """
    # k1: slope at the start of the interval  (worked for you)
    k1 = liouville_rhs(H, rho, hbar)

    # k2: slope at the midpoint, stepping halfway along k1
    k2 = ...

    # k3: slope at the midpoint, stepping halfway along k2
    k3 = ...

    # k4: slope at the end, stepping all the way along k3
    k4 = ...

    # Weighted average of the four slopes (RK4 combination)
    rho_next = ...
    return rho_next


In [ ]:
# With H = I, [H, rho] = 0, so the state does not change:
assert np.allclose(rk4_step(rho0, np.eye(8), 0.05), rho0)

# RK4 evolution preserves the trace of the density matrix:
assert np.isclose(np.trace(rk4_step(rho0, H1, 0.05)).real, 1.0)

# ...and preserves Hermiticity:
step = rk4_step(rho0, H1, 0.05)
assert np.allclose(step, step.conj().T)
print("✅ rk4_step tests passed!")


### ✍️ Required responses for Part 4

Answer each question in complete sentences.

1. If $H$ and $\rho$ are Hermitian, then $[H,\rho]$ is anti-Hermitian. Why is $-(i/\hbar)[H,\rho]$ Hermitian?

   **Response 1:** Replace this text with your answer.

2. What do $k_1$, $k_2$, $k_3$, and $k_4$ represent in RK4, and why are they combined using the weights $1,2,2,1$?

   **Response 2:** Replace this text with your answer.

3. What could go wrong if `dt` is too large? Would preservation of the trace alone prove that the simulation is accurate?

   **Response 3:** Replace this text with your answer.


# 🧮 Part 5: Sequential Dynamics and Bell-State Fidelity

## From Building Functions to Using Them

In Parts 1–4, you used the Design Recipe to build a reusable toolkit: `kron3`, `commutator`, `density_matrix`, `compute_expectation_value`, `liouville_rhs`, and `rk4_step`. In this final part, the emphasis shifts from writing new functions to deciding **which existing tool to use**, applying it correctly, and interpreting the results.

You will now use that toolkit to simulate the cavity-mediated entangling protocol: evolve the density matrix under $H_1$ and then $H_2$, track populations over time, and compare the final state with the target Bell state. In particular:

- `rk4_step` advances the density matrix;
- `density_matrix` constructs the target Bell-state density matrix;
- `compute_expectation_value` evaluates the fidelity because the target state is pure.


We will perform the propagation in two stages:

1. **Stage 1:** Evolve under $H_1$ for a duration $T_1=\pi/(4g)$, with qubit 1 coupled to the cavity and qubit 2 effectively off. This is a half-swap that creates a qubit-1–cavity superposition,

   $$
   |\Psi(T_1)\rangle=\frac{1}{\sqrt{2}}\left(|0eg\rangle-i|1gg\rangle\right).
   $$

2. **Stage 2:** Evolve under $H_2$ for a duration $T_2=\pi/(2g)$, with qubit 2 coupled and qubit 1 off. This stage completely swaps the **cavity component** into qubit 2,

   $$|1gg\rangle\longrightarrow-i|0ge\rangle,$$

   while the amplitude already residing on qubit 1 remains unchanged.

The full protocol therefore produces

$$
|\Psi(T_1+T_2)\rangle=\frac{1}{\sqrt{2}}\left(|0eg\rangle-|0ge\rangle\right),
$$

not a complete transfer of the entire excitation to qubit 2. At each time step, we will extract the diagonal elements of the density matrix, which are the populations of the uncoupled basis states.


In [ ]:
# 🕒 Time evolution of the system

# Choose step counts so that both switching times lie exactly on the time grid.
n_steps_1 = 200
n_steps_2 = 400
dt = T1_time / n_steps_1
assert np.isclose(dt, T2_time / n_steps_2)

n_steps = n_steps_1 + n_steps_2
t = np.arange(n_steps + 1) * dt

# Include the initial state at t = 0.
rho_current = np.copy(rho0)
pops_uncoupled = np.zeros((n_steps + 1, rho0.shape[0]))
pops_uncoupled[0, :] = np.real(np.diag(rho_current))

for i in range(n_steps):
    if i < n_steps_1:
        # Stage 1: qubit 1 is coupled to the cavity.
        H_stage = ...  # TODO: choose H1 or H2
    else:
        # Stage 2: qubit 2 is coupled to the cavity.
        H_stage = ...  # TODO: choose H1 or H2

    # TODO: Advance the density matrix by one RK4 step.
    rho_current = ...

    # TODO: Record all basis-state populations after this step.
    pops_uncoupled[i + 1, :] = ...

rho_final = np.copy(rho_current)

# --- ✅ Sanity checks ---
print("Simulation complete!")
print("Final time:", t[-1], " Expected:", T1_time + T2_time)
print("Shape of population array:", pops_uncoupled.shape)
print("Trace of final density matrix:", np.trace(rho_final))
print("Final density matrix Hermitian?", np.allclose(rho_final, rho_final.conj().T))

assert np.isclose(t[-1], T1_time + T2_time)
assert np.isclose(np.trace(rho_final).real, 1.0)
assert np.allclose(rho_final, rho_final.conj().T)


In [ ]:
# 🎨 Step: Visualize population dynamics during iSWAP evolution

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

# --- Plot populations for key states ---
plt.plot(t, pops_uncoupled[:, 2], '-',  lw=2.2, label=r"$|0eg\rangle$ (qubit 1 excited)")
plt.plot(t, pops_uncoupled[:, 4], '--', lw=2.2, label=r"$|1gg\rangle$ (cavity excited)")
plt.plot(t, pops_uncoupled[:, 1], ':',  lw=2.2, label=r"$|0ge\rangle$ (qubit 2 excited)")

# --- Aesthetic settings ---
plt.xlabel(r"Time $t$ (arb. units)", fontsize=13)
plt.ylabel(r"Population", fontsize=13)
plt.title("Population Dynamics in the Uncoupled Basis", fontsize=14, weight='bold')
plt.grid(alpha=0.3)
plt.legend(fontsize=11, loc="upper right", frameon=True)
plt.tight_layout()

# --- Optional: mark the gate switching points ---
plt.axvline(x=T1_time, color='gray', linestyle='--', alpha=0.6)
plt.text(T1_time + 0.2, 0.7, r"$T_1$", fontsize=12, color='gray')
plt.axvline(x=T1_time+ T2_time, color='gray', linestyle='--', alpha=0.6)
plt.text(T1_time + T2_time + 0.2, 0.7, r"$T_2$", fontsize=12, color='gray')

# --- Show the result ---
plt.show()


## 🎯 Comparing to the Target Bell State

At the end of the simulation, the cavity should return to $|0\rangle$ while the two molecular qubits occupy the entangled Bell state

$$
|\Psi_{\mathrm{target}}\rangle=
\frac{1}{\sqrt{2}}
\left(
|0eg\rangle-|0ge\rangle
\right).
$$

This target is the antisymmetric, singlet-like Bell state. Because the target is a pure state, its fidelity with the simulated final density matrix is

$$
F=\langle\Psi_{\mathrm{target}}|\rho_{\mathrm{final}}|\Psi_{\mathrm{target}}\rangle.
$$

Thus, the fidelity can be evaluated using the `compute_expectation_value` function developed in Part 3. A fidelity near 1 indicates that the protocol generated the intended Bell state.


In [ ]:
# 🧮 Compute fidelity with the target Bell state

# --- TODO 1: Construct the normalized target Bell state ---
# Hint: use the same relative sign and ordering as in the derived final state above.
target_ket = ...

# --- TODO 2: Form the target density matrix using your toolkit. ---
target_rho = ...

print("\nFinal density matrix (rho_final):")
print(np.array2string(rho_final, formatter={'all': complex_formatter}))

print("\nTarget density matrix (rho_target):")
print(np.array2string(target_rho, formatter={'all': complex_formatter}))

# --- TODO 3: For a pure target, fidelity is <target|rho_final|target>. ---
fidelity = ...

print(f"\n🔹 Fidelity with target Bell state: {fidelity:.6f}")
assert -1e-10 <= fidelity <= 1.0 + 1e-10


### ✍️ Required responses for Part 5

Answer each question in complete sentences and refer to your computed results.

1. Describe the dominant populations at $t=0$, $T_1$, and $T_1+T_2$. Distinguish the partial excitation amplitude placed in the cavity during Stage 1 from the complete transfer of that cavity component into qubit 2 during Stage 2.

   **Response 1:** Replace this text with your answer.

2. Why do the final populations alone not prove that the two qubits are entangled? What additional information is carried by the off-diagonal coherences and relative phase?

   **Response 2:** Replace this text with your answer.

3. What does the Bell-state fidelity measure? If it is less than 1, how could timing, time-step size, coupling parameters, or Hilbert-space truncation contribute?

   **Response 3:** Replace this text with your answer.
